In [1]:
import chipwhisperer as cw
import time, os
from Crypto.Cipher import AES
bitstream_path =  r'/home/sareeta/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit'
assert os.path.isfile(bitstream_path), f"Bitstream not found: {bitstream_path}"

# 2) Connect to the capture board (CWLite)
scope = cw.scope()
scope.default_setup()

# 3) Connect and Program the FPGA
print("Programming CW305 FPGA with:", bitstream_path)
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream_path, force=True)

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.54.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 1599282                   to 23284443                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0               
scope.clock.freq_ct

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/sareeta/chipwhisperer/software/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


In [2]:
# Fixed test vector
KEY = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
PT  = bytes.fromhex("00112233445566778899aabbccddeeff")

# Software AES (ground truth)
sw_ct = AES.new(KEY, AES.MODE_ECB).encrypt(PT)

# Send to FPGA
target.fpga_write(target.REG_CRYPT_KEY, KEY)
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)

hw_ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("=== NORMAL AES OPERATION ===")
print("Plaintext :", PT.hex())
print("Key       :", KEY.hex())
print("SW AES CT :", sw_ct.hex())
print("HW AES CT :", hw_ct.hex())
print("MATCH?    :", sw_ct == hw_ct)


=== NORMAL AES OPERATION ===
Plaintext : 00112233445566778899aabbccddeeff
Key       : 000102030405060708090a0b0c0d0e0f
SW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
HW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
MATCH?    : True


In [3]:
print(scope.trigger.triggers)
scope.trigger.triggers = "tio4"

tio4


In [6]:
print(scope.glitch.clk_src)
print(scope.clock.clkgen_freq)
print(scope.clock.adc_freq)

target
7384615.384615385
29538459


In [11]:
scope.dis()

(ChipWhisperer Scope ERROR|File naeusbchip.py:113) Scope already disconnected!


True

In [7]:
def aes_encrypt_once():
    # fire AES
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    # small wait so ciphertext register updates even if trigger missed
    time.sleep(0.001)
    return bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

def glitch_and_read():
    # arm scope + glitch
    scope.arm()

    # launch AES (this generates the external trigger used by ext_single)
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    # capture waveform (optional but useful for debugging alignment)
    cap_timeout = scope.capture()
    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    if cap_timeout:
        return "no_trigger", ct, None

    tr = scope.get_last_trace()

    if ct == hw_ct:
        return "correct", ct, tr
    else:
        return "faulty", ct, tr


In [8]:
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "glitch_only"
scope.glitch.repeat = 1
scope.io.glitch_lp = False
scope.io.glitch_hp = True
# scope.gain.gain = 46
# scope.gain.mode = "high"
# scope.adc.samples = 500
# scope.adc.offset = 0
# scope.adc.basic_mode = "rising_edge"
# scope.clock.adc_src = "clkgen_x4"
#scope.clock.freq_ctr_src = "clkgen"
scope.clock.adc_phase = 0
scope.trigger.triggers = "tio4"
scope.clock.extclk_freq = 10000000
scope.clock.clkgen_mul = 5
scope.clock.clkgen_div = 48
scope.clock.clkgen_freq = 10000000
scope.io.hs2 = "clkgen"
scope.glitch.clk_src = 'clkgen'
scope.glitch.ext_offset = 0
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired   "ext_single"  "continuous"
#scope.glitch.output = "clock_xor"
scope.glitch.width = 10
scope.glitch.offset = -20
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd'])
scope.glitch.repeat = 5   
print("Glitch ready.")
print("voltage glitch engine armed (but harmless so far)")


Glitch ready.
voltage glitch engine armed (but harmless so far)


In [9]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
ext_offsets = range(0, 3000, 20)
widths  = range(-49, 49, 1)
offsets = range(-49, 49, 1)

print("Starting Glitch Loop...")
for ext in ext_offsets:
    scope.glitch.ext_offset = ext
    for w in widths:
        scope.glitch.width = w
        for off in offsets:
            scope.glitch.offset = off
            
            for rep in range(N_PER_SETTING):
                # --- THE WORKING LOGIC FROM CODE 1 ---
                scope.arm()
                
                # Launch AES
                target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
                target.fpga_write(target.REG_CRYPT_GO, b"\x01")
                
                # Capture the hardware result
                cap_timeout = scope.capture()
                ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
                
                # Trace processing (from Code 2)
                tr = None
                if not cap_timeout:
                    tr = np.array(scope.get_last_trace(), dtype=np.float32)
                # --------------------------------------
    
                # Labeling Logic
                if cap_timeout:
                    label = "no_trigger"
                elif ct == hw_ct:
                    label = "correct"
                else:
                    label = "faulty"
                    faults.append((w, off, ct.hex()))
                    print(f"FAULT! w={w} off={off} ct={ct.hex()}")
    
                # Store in Hits dictionary
                hits[label] += 1
    
                # Create the record (from Code 2)
                records.append({
                    "ext":ext,
                    "width": w,
                    "offset": off,
                    "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                })
    
                # Store trace if it exists
                if tr is not None:
                    traces.append(tr)
                    labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=-49 off=-49 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-49 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-49 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-49 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-49 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-48 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-48 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-48 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-48 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-48 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-47 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-47 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-47 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-47 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-47 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-46 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-46 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! 

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! w=-49 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-5 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-5 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-5 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-5 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-5 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-4 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-4 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-4 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-4 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-4 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-3 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-3 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-3 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-3 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-3 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-49 off=-2 ct=f59d7cbf08fc47375511e6d9e

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! w=-48 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-48 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804


...

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! w=-11 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-2 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=-1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=0 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-11 off=1 ct=f59d7cbf08fc47375511e6d9eecb6804
FA

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


FAULT! w=-10 off=-9 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-9 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-9 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-8 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-8 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-8 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-8 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-8 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-7 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-7 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-7 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-7 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-7 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-6 ct=f59d7cbf08fc47375511e6d9eecb6804
FAULT! w=-10 off=-6 ct=f59d7cbf08fc47375511e6d9e

USBErrorIO: LIBUSB_ERROR_IO [-1]

In [6]:
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 1
#scope.io.glitch_hp = False 
# scope.gain.gain = 46 
# scope.gain.mode = "high" 
# scope.adc.samples = 500 
# scope.adc.offset = 0 
# scope.adc.basic_mode = "rising_edge" 
# scope.clock.adc_src = "clkgen_x4" 
scope.clock.freq_ctr_src = "clkgen" 
scope.clock.adc_phase = 0 
scope.trigger.triggers = "tio4" 
scope.clock.extclk_freq = 10000000 
scope.clock.clkgen_mul = 5 
scope.clock.clkgen_div = 48 
scope.clock.clkgen_freq = 10000000 
scope.io.hs2 = "glitch" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.ext_offset = 0 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.width = 10 
scope.glitch.offset = -20 
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd']) 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")

Glitch ready.
Clock glitch engine armed (but harmless so far)


In [7]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
widths  = range(-49, 49, 2)
offsets = range(-49, 49, 2)

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off
        
        for rep in range(N_PER_SETTING):
            # --- THE WORKING LOGIC FROM CODE 1 ---
            scope.arm()
            
            # Launch AES
            target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
            target.fpga_write(target.REG_CRYPT_GO, b"\x01")
            
            # Capture the hardware result
            cap_timeout = scope.capture()
            ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
            
            # Trace processing (from Code 2)
            tr = None
            if not cap_timeout:
                tr = np.array(scope.get_last_trace(), dtype=np.float32)
            # --------------------------------------

            # Labeling Logic
            if cap_timeout:
                label = "no_trigger"
            elif ct == hw_ct:
                label = "correct"
            else:
                label = "faulty"
                faults.append((w, off, ct.hex()))
                print(f"FAULT! w={w} off={off} ct={ct.hex()}")

            # Store in Hits dictionary
            hits[label] += 1

            # Create the record (from Code 2)
            records.append({
                "width": w,
                "offset": off,
                "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
            })

            # Store trace if it exists
            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=-47 off=-49 ct=93ad097b40728cb320a0a5fc32e3f3e1
FAULT! w=-39 off=1 ct=cd5183f73964bd13999f21f3cd0c4799
FAULT! w=-39 off=1 ct=6dc405736a7b0430d8cd1b1ba1b4935a
FAULT! w=-25 off=-49 ct=1acae18850538d7be827e71996cfab92
FAULT! w=-19 off=1 ct=0a8a093b89328caa90e0a5bcf2abf3a1
FAULT! w=-17 off=-49 ct=6944e0486a330430d8c532802020c452
FAULT! w=-15 off=-49 ct=256e093b5a328ca190e08576f28aa3a1
FAULT! w=1 off=-39 ct=6dc4aa736a7b0430d8cdb71ba1b4ad5a
FAULT! w=1 off=-3 ct=7024509000281420a0f81c14c0746056
FAULT! w=1 off=-3 ct=40a40050002800584090600084f03020
FAULT! w=1 off=-3 ct=a0e4800060a8207820a84871b234b471
FAULT! w=1 off=-3 ct=e0648000c0a8a06820a01871ba25b471
FAULT! w=1 off=-3 ct=00a820000000003000a0200022002024
FAULT! w=1 off=1 ct=e64ab59c24311faf6688b39ffb873205
FAULT! w=1 off=1 ct=84efc02bad2143fbe91b988d3e303826
FAULT! w=1 off=1 ct=60840f5281da2126c752811378cfe8bb
FAULT! w=1 off=1 ct=094480f80afb6418782d57201254a55a
FAULT! w=1 off=31 ct=3a7df3bed95c753bdb3293737

In [9]:
candidates = [
    (1, -3),
    (1, 1),
    (3, -1),
    (-39, 1),
]

In [10]:
from collections import Counter

def test_setting(w, off, ext_off=0, n=500):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct.hex()])
    print("Faulty:", len(cts) - counts[hw_ct.hex()])
    print("Top 10:")
    for ct, count in counts.most_common(10):
        tag = "CORRECT" if ct == hw_ct.hex() else "FAULT"
        print(count, tag, ct)

for w, off in candidates:
    test_setting(w, off)


SETTING: w= 1 off= -3 ext= 0
Timeouts: 0
Correct: 114
Faulty: 386
Top 10:
114 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
10 FAULT dc45a908a739e389b3b61333bf25bf71
9 FAULT dc45dd08a7b0e389b8b61333bf25bff7
5 FAULT 94713f2c36d3b301e51e0d96db073b41
5 FAULT d371c6a036d03d9551ad117de8503b64
4 FAULT 007119243611a613d36e0196d9203b7f
4 FAULT 94713ffd36d35901e5780d96c8073b41
4 FAULT d39d59a0e0e63d95e7ad11cce8508294
4 FAULT a371724e3635f9a1cf8b3060843d3b28
4 FAULT 724132ec544b3e6327bd07326f219345

SETTING: w= 1 off= 1 ext= 0
Timeouts: 0
Correct: 343
Faulty: 157
Top 10:
343 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
2 FAULT e103358aa5e522d1a12dab32c56958f2
2 FAULT e6f4689c3e921faff488b33bfb87ef61
1 FAULT b10196970a0971e8e6306d9a5d649394
1 FAULT b1d0d497885271e9d9536d295d449e90
1 FAULT a7946b95b289705e8bbd67b2ecf2611c
1 FAULT 222a43fbd8699ee74e8d39ff98386f10
1 FAULT 610a3990bdd4e9a4dc9c6a4e23ff7543
1 FAULT e6a1c98ae10615af792379488a8f714b
1 FAULT f659934c780c71a01252bee05df7a4b0

SETTING: w= 3 off= -1

In [13]:
from collections import Counter
import time
import pandas as pd

GOLDEN = hw_ct.hex()

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)
    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

def test_setting(w, off, ext=0, n=500):
    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    for rep in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        time.sleep(0.001)
        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    faulty_counts = {ct: c for ct, c in counts.items() if ct != GOLDEN}

    if faulty_counts:
        top_fault_ct, top_fault_count = Counter(faulty_counts).most_common(1)[0]
        changed = diff_bytes(top_fault_ct)
    else:
        top_fault_ct = None
        top_fault_count = 0
        changed = []

    result = {
        "ext_offset": ext,
        "width": w,
        "offset": off,
        "trials": n,
        "valid_captures": len(cts),
        "timeouts": timeouts,
        "correct": counts[GOLDEN],
        "faulty": len(cts) - counts[GOLDEN],
        "top_fault_count": top_fault_count,
        "top_fault_ct": top_fault_ct,
        "changed_byte_count": len(changed),
        "changed_bytes": changed,
    }

    return result, counts


# Focus around your best point: w=-39, off=1, ext=0
results = []
all_counts = {}

ext_offsets = range(0, 10)
widths = range(-43, -34, 1)
offsets = range(-3, 6, 1)

N = 500

print("Starting focused repeatability scan...")

for ext in ext_offsets:
    for w in widths:
        for off in offsets:
            result, counts = test_setting(w, off, ext=ext, n=N)

            results.append(result)
            all_counts[(ext, w, off)] = counts

            if result["top_fault_count"] > 0:
                print(
                    f"ext={ext:2d} w={w:4d} off={off:3d} | "
                    f"correct={result['correct']:3d} faulty={result['faulty']:3d} "
                    f"top_fault={result['top_fault_count']:3d} "
                    f"changed_bytes={result['changed_byte_count']} "
                    f"ct={result['top_fault_ct']}"
                )

df = pd.DataFrame(results)

# Ranking: repeated fault first, then fewer changed bytes, then fewer timeouts
df_ranked = df.sort_values(
    by=["top_fault_count", "changed_byte_count", "timeouts"],
    ascending=[False, True, True]
)

print("\nTop candidate settings:")
print(df_ranked.head(20)[[
    "ext_offset",
    "width",
    "offset",
    "correct",
    "faulty",
    "timeouts",
    "top_fault_count",
    "changed_byte_count",
    "changed_bytes",
    "top_fault_ct"
]])

Starting focused repeatability scan...


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -43 off=  1 | correct=494 faulty=  6 top_fault=  1 changed_bytes=16 ct=1789ff7634b6eb01795c14a6233fe37f


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -42 off=  1 | correct=494 faulty=  6 top_fault=  1 changed_bytes=16 ct=085183763b64eb019b5c14f3213f4799


...

ext= 2 w= -38 off=  1 | correct=499 faulty=  1 top_fault=  1 changed_bytes=16 ct=c265ac3c67c818575be06812f50f9ea8


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 2 w= -37 off=  1 | correct=499 faulty=  1 top_fault=  1 changed_bytes=4 ct=69c4e0d86a3b0420d8cda68070b4c552


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfigu


Top candidate settings:
     ext_offset  width  offset  correct  faulty  timeouts  top_fault_count  \
130           1    -38       1      492       8         0                4   
31            0    -40       1      492       8         0                3   
139           1    -37       1      498       2         0                2   
202           2    -39       1      495       5         0                2   
22            0    -41       1      495       5         0                2   
85            1    -43       1      493       7         0                2   
121           1    -39       1      493       7         0                2   
112           1    -40       1      494       6         0                2   
175           2    -42       1      496       4         0                2   
166           2    -43       1      494       6         0                1   
220           2    -37       1      499       1         0                1   
193           2    -40       1      495

In [14]:
best = df_ranked.iloc[0]

ext = int(best["ext_offset"])
w = int(best["width"])
off = int(best["offset"])

print("Best setting:", ext, w, off)

result, counts = test_setting(w, off, ext=ext, n=3000)

print(result)

print("\nTop 20 ciphertexts:")
for ct, count in counts.most_common(20):
    tag = "CORRECT" if ct == GOLDEN else "FAULT"
    print(count, tag, ct, diff_bytes(ct) if ct != GOLDEN else [])

Best setting: 1 -38 1
{'ext_offset': 1, 'width': -38, 'offset': 1, 'trials': 3000, 'valid_captures': 3000, 'timeouts': 0, 'correct': 2988, 'faulty': 12, 'top_fault_count': 4, 'top_fault_ct': '6dc405736a7b0430d8cd1b1ba1b4935a', 'changed_byte_count': 7, 'changed_bytes': [0, 2, 3, 10, 11, 12, 14]}

Top 20 ciphertexts:
2988 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a []
4 FAULT 6dc405736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
2 FAULT 6dc4e0d86a7b0430d8cdb78070b4c55a [0]
1 FAULT 6dc418736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
1 FAULT 2ef71aa7417fe3f7327dcb1b503864da [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007755f4df24c487324dcb949438e663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 6dc4e0736a7b0430d8cdb71ba1b4bf5a [0, 3, 11, 12, 14]
1 FAULT 007755fedf01c4873229cb845038e4db [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007754f0df24c5918bba5396dd3ce663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [15]:
from collections import Counter
import time

GOLDEN = hw_ct.hex()

candidates = [
    (1, -38, 1),   # ext, width, offset
    (0, -40, 1),
    (2, -39, 1),
]

N = 5000

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)

    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

for ext, w, off in candidates:

    print("\n" + "="*80)
    print(f"TESTING ext={ext}, width={w}, offset={off}")
    print("="*80)

    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    start = time.time()

    for i in range(N):

        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

        if (i + 1) % 500 == 0:
            print(f"{i+1}/{N}")

    elapsed = time.time() - start

    counts = Counter(cts)

    correct = counts[GOLDEN]
    faulty = len(cts) - correct

    print("\nSUMMARY")
    print("Elapsed:", round(elapsed, 2), "seconds")
    print("Timeouts:", timeouts)
    print("Correct:", correct)
    print("Faulty :", faulty)

    print("\nTOP 20 CIPHERTEXTS")

    for ct, count in counts.most_common(20):

        if ct == GOLDEN:
            print(f"{count:5d}  CORRECT  {ct}")
        else:
            changed = diff_bytes(ct)

            print(
                f"{count:5d}  FAULT    {ct} "
                f"changed_bytes={len(changed)} "
                f"indices={changed}"
            )


TESTING ext=1, width=-38, offset=1
500/5000
1000/5000
1500/5000
2000/5000
2500/5000
3000/5000
3500/5000
4000/5000
4500/5000
5000/5000

SUMMARY
Elapsed: 15.76 seconds
Timeouts: 0
Correct: 4952
Faulty : 48

TOP 20 CIPHERTEXTS
 4952  CORRECT  69c4e0d86a7b0430d8cdb78070b4c55a
   14  FAULT    6dc4e0d86a7b0430d8cdb78070b4c55a changed_bytes=1 indices=[0]
    6  FAULT    6dc419736a7b0430d8cdb71ba1b4935a changed_bytes=6 indices=[0, 2, 3, 11, 12, 14]
    4  FAULT    6dc405736a7b0430d8cd1b1ba1b4935a changed_bytes=7 indices=[0, 2, 3, 10, 11, 12, 14]
    3  FAULT    007754f0cf24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7b0430d9cd1b1ba0b4935a changed_bytes=8 indices=[0, 2, 3, 8, 10, 11, 12, 14]
    2  FAULT    007754f0df24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7be330d9aa1b1ba083615a changed_bytes=11 indices=[0, 2, 3, 6, 8, 9, 10